# 面试题：如何从零实现 TextCNN 与 HAN，并解释它们分别适合什么文本？

## 面试回答主线

TextCNN 把连续 n-gram 当作局部模式，用不同窗口的卷积核提取“无法登录”“退款到账”等短语，再做全局最大池化；它并行、便宜，适合关键词和局部语序决定标签的短文本。HAN 先在句内对词做注意力汇聚，再在文档内对句子做注意力汇聚，因此能给出“哪句话、哪个词推动了分类”的两级证据，更适合多句文档。工程上不能只看最终 accuracy：必须检查 padding mask、注意力是否集中在真实句子、类别混淆和长度分桶。下面不用 `nn.Conv1d`、RNN、现成 TextCNN/HAN 或 Trainer，而是用参数矩阵、`unfold`、softmax 和反向传播手写核心机制。

## 真实案例：中文客服工单路由

我们构造 12 条脱敏式客服工单，每条由两句话组成，标签是“退款 / 物流 / 账号”。句子保留订单、时间、动作和否定等业务语义；它们是可离线复现的教学样本，不是真实用户数据，也不能代表线上泛化。实验会在同一小数据上观察拟合能力，生产结论必须另做时间切分与盲测。

In [1]:
import math  # 导入平方根用于参数初始化缩放。
import warnings  # 导入告警控制工具以保持保存输出聚焦教学结果。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)  # 只过滤当前环境在导入 PyTorch 时产生的第三方弃用提示。
import torch  # 导入 PyTorch 以手写网络并执行真实反向传播。
from torch import nn  # 导入最基础的模块与可学习参数容器。
import torch.nn.functional as F  # 导入底层交叉熵与 padding 操作。
torch.set_num_threads(1)  # 小型教学张量固定单线程以减少运行波动。
torch.manual_seed(17)  # 固定数据噪声、参数初始化与训练结果。
records = [  # 构造带两句话正文与业务标签的客服工单。
    (["订单 已经 取消", "退款 三天 还没 到账"], "退款"),  # 退款未到账工单。
    (["商品 已经 退回", "原路 退款 一直 没有"], "退款"),  # 退货后退款工单。
    (["银行卡 已经 更换", "退款 要 改到 余额"], "退款"),  # 退款渠道变更工单。
    (["不是 催 物流", "我要 查询 退款 进度"], "退款"),  # 含物流干扰词的退款工单。
    (["快递 显示 签收", "但是 包裹 没有 收到"], "物流"),  # 虚假签收工单。
    (["订单 发货 五天", "物流 信息 一直 不动"], "物流"),  # 物流停滞工单。
    (["地址 填写 错误", "包裹 能否 修改 派送"], "物流"),  # 修改派送地址工单。
    (["退款 已经 到账", "现在 只想 催 快递"], "物流"),  # 含退款干扰词的物流工单。
    (["手机号码 已经 停用", "账号 无法 接收 验证码"], "账号"),  # 验证码登录工单。
    (["连续 输入 密码", "账号 突然 被 锁定"], "账号"),  # 密码锁定工单。
    (["新手机 登录", "提示 需要 身份 验证"], "账号"),  # 新设备验证工单。
    (["订单 和 物流 都 正常", "只是 账号 无法 登录"], "账号"),  # 含订单物流干扰词的账号工单。
]  # 完成十二条多句工单。
labels = ["退款", "物流", "账号"]  # 固定三类标签的展示与编码顺序。
label_to_id = {label: index for index, label in enumerate(labels)}  # 建立标签到整数的映射。
print("编号  标签  句子一                     句子二")  # 输出输入预览表头。
for index, (sentences, label) in enumerate(records, start=1):  # 逐条遍历真实语义工单。
    print(f"{index:>2}    {label}  {sentences[0]:<25} {sentences[1]}")  # 展示正文层级与监督标签。

编号  标签  句子一                     句子二
 1    退款  订单 已经 取消                  退款 三天 还没 到账
 2    退款  商品 已经 退回                  原路 退款 一直 没有
 3    退款  银行卡 已经 更换                 退款 要 改到 余额
 4    退款  不是 催 物流                   我要 查询 退款 进度
 5    物流  快递 显示 签收                  但是 包裹 没有 收到
 6    物流  订单 发货 五天                  物流 信息 一直 不动
 7    物流  地址 填写 错误                  包裹 能否 修改 派送
 8    物流  退款 已经 到账                  现在 只想 催 快递
 9    账号  手机号码 已经 停用                账号 无法 接收 验证码
10    账号  连续 输入 密码                  账号 突然 被 锁定
11    账号  新手机 登录                    提示 需要 身份 验证
12    账号  订单 和 物流 都 正常              只是 账号 无法 登录


## Baseline（基线）：固定关键词首次命中

最便宜的路由器按“退款 → 物流 → 账号”的固定顺序找关键词。这能解释、也无需训练，但只要工单同时提到多个领域或出现否定词，就会把干扰背景当成真实诉求。

In [2]:
keyword_order = [("退款", ["退款", "退回"]), ("物流", ["物流", "快递", "包裹"]), ("账号", ["账号", "登录", "密码", "验证码"])]  # 定义有顺序偏置的关键词规则。
baseline_predictions = []  # 收集逐工单规则预测。
for sentences, _ in records:  # 遍历全部工单正文。
    text = " ".join(sentences)  # 把两句话拼成规则可扫描的字符串。
    prediction = "账号"  # 未命中时回退到账号队列。
    for candidate, keywords in keyword_order:  # 按固定优先级检查类别词表。
        if any(keyword in text for keyword in keywords):  # 判断当前类别是否出现任一词。
            prediction = candidate  # 使用首次命中的类别作为输出。
            break  # 停止后续类别扫描以模拟常见规则路由。
    baseline_predictions.append(prediction)  # 保存当前工单预测。
baseline_correct = [prediction == gold for prediction, (_, gold) in zip(baseline_predictions, records)]  # 比较预测与人工标签。
baseline_accuracy = sum(baseline_correct) / len(baseline_correct)  # 计算规则准确率。
print("编号  gold  规则预测  正确  触发说明")  # 输出逐样本基线结果表头。
for index, ((sentences, gold), prediction, correct) in enumerate(zip(records, baseline_predictions, baseline_correct), start=1):  # 对齐输入、预测和判定。
    trigger = "含多领域词" if len({label for label, words in keyword_order if any(word in " ".join(sentences) for word in words)}) > 1 else "单领域词"  # 标记规则最容易混淆的多领域样本。
    print(f"{index:>2}    {gold:<2}    {prediction:<2}     {str(correct):<5} {trigger}")  # 展示规则在哪里被干扰词欺骗。
print(f"关键词首次命中准确率：{baseline_accuracy:.1%}")  # 输出与神经网络同口径的基线指标。

编号  gold  规则预测  正确  触发说明
 1    退款    退款     True  单领域词
 2    退款    退款     True  单领域词
 3    退款    退款     True  单领域词
 4    退款    退款     True  含多领域词
 5    物流    物流     True  单领域词
 6    物流    物流     True  单领域词
 7    物流    物流     True  单领域词
 8    物流    退款     False 含多领域词
 9    账号    账号     True  单领域词
10    账号    账号     True  单领域词
11    账号    账号     True  单领域词
12    账号    物流     False 含多领域词
关键词首次命中准确率：83.3%


## 核心实现一：张量化文档与手写 TextCNN

每篇文档先拼成一条 token 序列。`unfold` 显式取出所有宽度为 2/3 的窗口，随后与卷积核参数做 `einsum`，这正是 1D 卷积的核心；每个卷积核再对时间维取最大值。词向量参数表的第 0 行可以是任意值，但 forward 会把 PAD 查表结果乘零，因此 PAD 梯度也被截断。任何包含一个或多个 PAD 的不完整 n-gram 都在池化前屏蔽，短文本补得更长不能制造长度伪特征。

In [3]:
all_tokens = sorted({token for sentences, _ in records for sentence in sentences for token in sentence.split()})  # 从教学数据建立可检查词表。
token_to_id = {"[PAD]": 0, "[UNK]": 1}  # 预留 padding 与未知词编号。
for token in all_tokens:  # 按稳定顺序加入业务词。
    token_to_id[token] = len(token_to_id)  # 为当前词分配唯一递增编号。
flat_tokens = [[token for sentence in sentences for token in sentence.split()] for sentences, _ in records]  # 把每篇两句文档展平成 TextCNN 输入。
max_tokens = max(len(tokens) for tokens in flat_tokens)  # 读取批内最大 token 长度。
text_tensor = torch.zeros(len(records), max_tokens, dtype=torch.long)  # 创建右侧 padding 的整数张量。
for row, tokens in enumerate(flat_tokens):  # 遍历每篇展平文档。
    ids = [token_to_id.get(token, 1) for token in tokens]  # 把业务词映射为词表编号。
    text_tensor[row, :len(ids)] = torch.tensor(ids)  # 把真实 token 写入对应前缀。
target_tensor = torch.tensor([label_to_id[label] for _, label in records])  # 编码三类监督标签。
class ManualTextCNN(nn.Module):  # 定义不用 nn.Conv1d 的 TextCNN 分类器。
    def __init__(self, vocabulary_size, embedding_dim, filters_per_width, class_count):  # 初始化 embedding、卷积核和分类头。
        super().__init__()  # 注册 PyTorch 模块状态。
        self.embedding = nn.Parameter(torch.randn(vocabulary_size, embedding_dim) * 0.12)  # 创建可学习词向量表。
        self.kernel_widths = (2, 3)  # 使用二元与三元短语窗口。
        self.kernels = nn.ParameterList([nn.Parameter(torch.randn(filters_per_width, width, embedding_dim) * 0.12) for width in self.kernel_widths])  # 为每种窗口创建卷积核。
        self.biases = nn.ParameterList([nn.Parameter(torch.zeros(filters_per_width)) for _ in self.kernel_widths])  # 为每组卷积核创建偏置。
        self.output_weight = nn.Parameter(torch.randn(filters_per_width * len(self.kernel_widths), class_count) * 0.12)  # 创建池化特征到类别的权重。
        self.output_bias = nn.Parameter(torch.zeros(class_count))  # 创建类别偏置。
    def forward(self, token_ids, return_features=False):  # 执行 embedding、手写卷积、池化和分类。
        padding_mask = token_ids.eq(0)  # 标记右侧 PAD 位置。
        embedding_mask = token_ids.ne(0).unsqueeze(-1).to(self.embedding.dtype)  # 把有效 token mask 扩展到词向量维度。
        embeddings = self.embedding[token_ids] * embedding_mask  # 在 forward 中把 PAD 查表结果恒置零并阻断第零行梯度。
        pooled_features = []  # 收集每种窗口的最大池化结果。
        activation_maps = []  # 保留卷积激活以便解释中间过程。
        for width, kernel, bias in zip(self.kernel_widths, self.kernels, self.biases):  # 逐种 n-gram 宽度执行手写卷积。
            windows = embeddings.unfold(1, width, 1).permute(0, 1, 3, 2)  # 取出 batch、位置、窗口、维度的局部块。
            scores = torch.einsum("blwe,fwe->blf", windows, kernel) + bias  # 让每个局部窗口与每个卷积核做内积。
            invalid = padding_mask.unfold(1, width, 1).any(dim=-1)  # 屏蔽包含任意 PAD 的不完整 n-gram 窗口。
            scores = scores.masked_fill(invalid.unsqueeze(-1), -1e4)  # 防止部分或全部 padding 形成长度伪特征。
            activations = torch.relu(scores)  # 对卷积分数应用 ReLU 非线性。
            pooled_features.append(activations.max(dim=1).values)  # 每个卷积核保留最强短语响应。
            activation_maps.append(activations)  # 保存完整位置响应供案例观察。
        features = torch.cat(pooled_features, dim=-1)  # 拼接不同 n-gram 尺度的文档表示。
        logits = features @ self.output_weight + self.output_bias  # 计算三类未归一化分数。
        if return_features:  # 解释模式需要同时返回局部激活。
            return logits, features, activation_maps  # 返回分类结果与中间过程。
        return logits  # 普通训练模式只返回类别分数。
torch.manual_seed(23)  # 固定 TextCNN 参数初始化。
textcnn = ManualTextCNN(len(token_to_id), 12, 10, len(labels))  # 创建词维十二、每宽十核的模型。
preview_logits, preview_features, preview_maps = textcnn(text_tensor[:2], return_features=True)  # 对两条工单执行一次前向传播。
print("词表大小 / 输入形状：", len(token_to_id), tuple(text_tensor.shape))  # 展示离散输入规模与 padding 后形状。
print("宽度2激活 / 宽度3激活 / 拼接特征：", tuple(preview_maps[0].shape), tuple(preview_maps[1].shape), tuple(preview_features.shape))  # 展示卷积机制的真实张量变化。
print("首条工单宽度2最强响应：", round(float(preview_maps[0][0].max()), 4))  # 输出一个可追踪的卷积中间值。

词表大小 / 输入形状： 65 (12, 9)
宽度2激活 / 宽度3激活 / 拼接特征： (2, 8, 10) (2, 7, 10) (2, 20)
首条工单宽度2最强响应： 0.167


## 核心实现二：手写两级注意力 HAN

为了把重点放在层级聚合，这里用可学习投影代替原论文的双向 GRU：词级先计算每个词对上下文向量的相似度，带 mask softmax 后得到句向量；句级重复同一过程得到文档向量。HAN 与 TextCNN 使用同一条 PAD 合同：查表后先把 PAD 向量乘零以阻断 embedding 第 0 行梯度，再让两级注意力中的 PAD 权重严格为 0。

In [4]:
sentence_tokens = [[sentence.split() for sentence in sentences] for sentences, _ in records]  # 保留文档到句子再到词的两级结构。
max_sentences = max(len(document) for document in sentence_tokens)  # 读取每篇文档最大句子数。
max_words = max(len(sentence) for document in sentence_tokens for sentence in document)  # 读取每句话最大词数。
hierarchy_tensor = torch.zeros(len(records), max_sentences, max_words, dtype=torch.long)  # 创建带两层 padding 的文档张量。
for document_index, document in enumerate(sentence_tokens):  # 遍历每篇层级文档。
    for sentence_index, sentence in enumerate(document):  # 遍历文档中的每句话。
        ids = [token_to_id.get(token, 1) for token in sentence]  # 把句内词转换为编号。
        hierarchy_tensor[document_index, sentence_index, :len(ids)] = torch.tensor(ids)  # 写入真实词并保留右侧 PAD。
class ManualHAN(nn.Module):  # 定义词级与句级两层注意力分类器。
    def __init__(self, vocabulary_size, embedding_dim, hidden_dim, class_count):  # 初始化两级投影、上下文向量与分类头。
        super().__init__()  # 注册 PyTorch 模块状态。
        self.embedding = nn.Parameter(torch.randn(vocabulary_size, embedding_dim) * 0.12)  # 创建可学习词向量。
        self.word_weight = nn.Parameter(torch.randn(embedding_dim, hidden_dim) * 0.12)  # 创建词级非线性投影。
        self.word_context = nn.Parameter(torch.randn(hidden_dim) * 0.12)  # 创建词级注意力查询向量。
        self.sentence_weight = nn.Parameter(torch.randn(embedding_dim, hidden_dim) * 0.12)  # 创建句级非线性投影。
        self.sentence_context = nn.Parameter(torch.randn(hidden_dim) * 0.12)  # 创建句级注意力查询向量。
        self.output_weight = nn.Parameter(torch.randn(embedding_dim, class_count) * 0.12)  # 创建文档表示到类别的权重。
        self.output_bias = nn.Parameter(torch.zeros(class_count))  # 创建类别偏置。
    def masked_attention(self, values, valid_mask, weight, context):  # 手写投影、打分、mask softmax 与加权和。
        hidden = torch.tanh(values @ weight)  # 把每个候选映射到注意力隐藏空间。
        scores = hidden @ context  # 计算每个候选与上下文向量的匹配分数。
        scores = scores.masked_fill(~valid_mask, -1e4)  # 让 PAD 在 softmax 前得到极小分数。
        attention = torch.softmax(scores, dim=-1)  # 先得到数值稳定的候选注意力权重。
        attention = attention * valid_mask.to(attention.dtype)  # softmax 后再次乘原始 mask，使全空行的 PAD 权重也严格为零。
        normalizer = attention.sum(dim=-1, keepdim=True)  # 计算每行剩余有效候选的概率质量。
        attention = attention / normalizer.clamp_min(1e-8)  # 非空行重新归一化，全空行保持全零且不会产生 NaN。
        summary = (attention.unsqueeze(-1) * values).sum(dim=-2)  # 按修正后的权重汇聚词或句子表示。
        return summary, attention  # 返回聚合表示与可解释权重。
    def forward(self, document_ids, return_attention=False):  # 对 batch、句、词三维输入执行两级汇聚。
        word_mask = document_ids.ne(0)  # 标出每句话中的真实词位置。
        embedding_mask = word_mask.unsqueeze(-1).to(self.embedding.dtype)  # 把词有效性 mask 扩展到 embedding 维度。
        embeddings = self.embedding[document_ids] * embedding_mask  # 让 PAD 查表结果恒为零并阻断第零行梯度。
        batch_size, sentence_count, word_count, embedding_dim = embeddings.shape  # 读取层级张量各维大小。
        flat_embeddings = embeddings.reshape(batch_size * sentence_count, word_count, embedding_dim)  # 合并批量和句子以并行做词注意力。
        flat_word_mask = word_mask.reshape(batch_size * sentence_count, word_count)  # 同步展平词有效性 mask。
        flat_sentence_vectors, flat_word_attention = self.masked_attention(flat_embeddings, flat_word_mask, self.word_weight, self.word_context)  # 用原始 mask 汇聚每句话并让全空句权重全零。
        sentence_vectors = flat_sentence_vectors.reshape(batch_size, sentence_count, embedding_dim)  # 恢复 batch、句子、维度结构。
        sentence_mask = word_mask.any(dim=-1)  # 只把至少有一个真实词的句子视为有效。
        document_vectors, sentence_attention = self.masked_attention(sentence_vectors, sentence_mask, self.sentence_weight, self.sentence_context)  # 汇聚文档内句子表示。
        logits = document_vectors @ self.output_weight + self.output_bias  # 计算最终三类分数。
        if return_attention:  # 解释模式需要返回两级注意力。
            word_attention = flat_word_attention.reshape(batch_size, sentence_count, word_count)  # 恢复词权重的层级形状。
            return logits, word_attention, sentence_attention  # 返回预测与词句两级证据。
        return logits  # 普通训练模式只返回分类分数。
torch.manual_seed(29)  # 固定 HAN 参数初始化。
han = ManualHAN(len(token_to_id), 12, 10, len(labels))  # 创建十二维词向量的两级注意力模型。
han_logits, word_attention, sentence_attention = han(hierarchy_tensor[:2], return_attention=True)  # 对两篇工单观察未训练前向过程。
print("层级输入 / 词权重 / 句权重：", tuple(hierarchy_tensor.shape), tuple(word_attention.shape), tuple(sentence_attention.shape))  # 展示两级输入输出形状。
print("首篇两句注意力和：", round(float(sentence_attention[0].sum()), 6))  # 验证真实句子权重归一化但不使用断言代替观察。

层级输入 / 词权重 / 句权重： (12, 2, 5) (2, 2, 5) (2, 2)
首篇两句注意力和： 1.0


## 真实训练与结果表

下面对两套手写模型分别执行 Adam、交叉熵、`zero_grad → backward → step`。因为只有 12 条样本，结果表示“实现能否学到这批模式”，不表示线上泛化能力。除了最终准确率，还打印首轮与末轮 loss、梯度范数及每条工单预测。

In [5]:
def train_classifier(model, inputs, epochs=220):  # 定义共享的小数据真实训练循环。
    optimizer = torch.optim.Adam(model.parameters(), lr=0.035)  # 创建直接更新手写参数的 Adam 优化器。
    history = []  # 保存 loss、accuracy 与梯度范数轨迹。
    for epoch in range(epochs):  # 在全部教学样本上重复优化。
        optimizer.zero_grad()  # 清除上一轮累计梯度。
        logits = model(inputs)  # 调用手写 forward 计算三类分数。
        loss = F.cross_entropy(logits, target_tensor)  # 计算多类负对数似然。
        loss.backward()  # 从分类误差反向传播到卷积核或注意力参数。
        gradient_norm = torch.sqrt(sum((parameter.grad ** 2).sum() for parameter in model.parameters() if parameter.grad is not None))  # 汇总当前轮全参数梯度二范数。
        optimizer.step()  # 根据真实梯度更新全部参数。
        accuracy = (logits.argmax(dim=-1) == target_tensor).float().mean()  # 计算当前轮训练准确率。
        history.append((float(loss.detach()), float(accuracy.detach()), float(gradient_norm.detach())))  # 保存可解释训练轨迹。
    return history  # 返回训练过程供对比而非只给最终布尔值。
textcnn_history = train_classifier(textcnn, text_tensor)  # 真实训练手写 TextCNN。
han_history = train_classifier(han, hierarchy_tensor)  # 真实训练手写 HAN。
with torch.no_grad():  # 关闭评估阶段梯度记录。
    textcnn_probabilities = torch.softmax(textcnn(text_tensor), dim=-1)  # 计算 TextCNN 类别概率。
    han_scores, trained_word_attention, trained_sentence_attention = han(hierarchy_tensor, return_attention=True)  # 获取 HAN 概率前分数与两级注意力。
    han_probabilities = torch.softmax(han_scores, dim=-1)  # 把 HAN 分数转换为类别概率。
textcnn_predictions = textcnn_probabilities.argmax(dim=-1)  # 取 TextCNN 最大概率类别。
han_predictions = han_probabilities.argmax(dim=-1)  # 取 HAN 最大概率类别。
textcnn_accuracy = float((textcnn_predictions == target_tensor).float().mean())  # 计算 TextCNN 同数据准确率。
han_accuracy = float((han_predictions == target_tensor).float().mean())  # 计算 HAN 同数据准确率。
print("模型       首轮loss  末轮loss  首轮梯度  末轮准确率")  # 输出训练过程对比表头。
print(f"TextCNN    {textcnn_history[0][0]:.4f}    {textcnn_history[-1][0]:.4f}    {textcnn_history[0][2]:.4f}     {textcnn_accuracy:.1%}")  # 展示 TextCNN 优化证据。
print(f"HAN        {han_history[0][0]:.4f}    {han_history[-1][0]:.4f}    {han_history[0][2]:.4f}     {han_accuracy:.1%}")  # 展示 HAN 优化证据。
print("编号 gold 规则 TextCNN HAN  TextCNN置信度 HAN置信度")  # 输出逐样本预测结果表头。
for index, ((_, gold), rule, text_id, han_id) in enumerate(zip(records, baseline_predictions, textcnn_predictions.tolist(), han_predictions.tolist()), start=1):  # 对齐三种方案的输出。
    print(f"{index:>2}   {gold:<2} {rule:<2}   {labels[text_id]:<2}      {labels[han_id]:<2}    {float(textcnn_probabilities[index - 1, text_id]):.3f}         {float(han_probabilities[index - 1, han_id]):.3f}")  # 展示预测和模型置信度。
focus_index = 11  # 选择同时含订单、物流与账号词的第十二条反例。
print("第12条句级注意力：", [round(float(value), 3) for value in trained_sentence_attention[focus_index]])  # 展示模型如何分配两句话证据。
for sentence_index, sentence in enumerate(sentence_tokens[focus_index]):  # 遍历该工单的两句话。
    weights = trained_word_attention[focus_index, sentence_index, :len(sentence)]  # 截取真实词的注意力权重。
    print("句", sentence_index + 1, list(zip(sentence, [round(float(value), 3) for value in weights])))  # 对齐输出词与其贡献权重。

模型       首轮loss  末轮loss  首轮梯度  末轮准确率
TextCNN    1.1050    0.0000    0.1431     100.0%
HAN        1.1055    0.0000    0.0573     100.0%
编号 gold 规则 TextCNN HAN  TextCNN置信度 HAN置信度
 1   退款 退款   退款      退款    1.000         1.000
 2   退款 退款   退款      退款    1.000         1.000
 3   退款 退款   退款      退款    1.000         1.000
 4   退款 退款   退款      退款    1.000         1.000
 5   物流 物流   物流      物流    1.000         1.000
 6   物流 物流   物流      物流    1.000         1.000
 7   物流 物流   物流      物流    1.000         1.000
 8   物流 退款   物流      物流    1.000         1.000
 9   账号 账号   账号      账号    1.000         1.000
10   账号 账号   账号      账号    1.000         1.000
11   账号 账号   账号      账号    1.000         1.000
12   账号 物流   账号      账号    1.000         1.000
第12条句级注意力： [0.617, 0.383]
句 1 [('订单', 0.0), ('和', 0.424), ('物流', 0.0), ('都', 0.568), ('正常', 0.008)]
句 2 [('只是', 0.374), ('账号', 0.302), ('无法', 0.288), ('登录', 0.036)]


## 结果解读

固定关键词会把“不是催物流”“退款已到账”等背景词当诉求，因此只有部分样本正确。两个手写网络的 loss 都真实下降，且在这 12 条教学样本上学会分类；TextCNN 的中间张量显示每个卷积核在所有二元/三元窗口上扫描，HAN 则能直接查看第 12 条工单中“只是账号无法登录”这一句及句内词的权重。这里的 100% 是小样本拟合检查，不能声称对新写法泛化。

## 失败案例与修正：PAD 参数、部分窗口和补齐长度泄漏

mask 不是“shape 对了就行”。若只屏蔽全 PAD 窗口，`真实词 + PAD` 仍能被 TextCNN 当成短语；若只给注意力分数加 mask，随机可训练的 PAD embedding 仍会进入前置投影。下面先复现空句抢走注意力，再直接修改两套模型的 PAD 参数为巨大哨兵值、增加右侧词/句 padding，并执行 backward。额外探针会分别输入“正常文档中新添的全 PAD 句”和“整篇全空文档”，验证词级与句级注意力全零且所有输出有限。正确实现应同时满足：有效语义 logits 与池化特征不变、有效注意力不变、PAD 参数修改无效、embedding 第 0 行梯度严格为零。

In [6]:
raw_sentence_scores = torch.tensor([[0.4, 0.2, 4.0]])  # 模拟两句正文加一句高分 PAD 的注意力打分。
valid_sentences = torch.tensor([[True, True, False]])  # 明确第三个位置没有真实句子。
naive_attention = torch.softmax(raw_sentence_scores, dim=-1)  # 错误做法直接对所有位置归一化。
masked_scores = raw_sentence_scores.masked_fill(~valid_sentences, -1e4)  # 正确做法在 softmax 前屏蔽 PAD。
fixed_attention = torch.softmax(masked_scores, dim=-1)  # 对剩余真实句子重新归一化。
real_sentence_value = torch.tensor([[2.0, -1.0, 99.0]])  # 让 PAD 携带一个能严重污染文档表示的哨兵值。
naive_summary = (naive_attention * real_sentence_value).sum(dim=-1)  # 计算错误注意力下的文档标量。
fixed_summary = (fixed_attention * real_sentence_value).sum(dim=-1)  # 计算 mask 修复后的文档标量。
semantic_length = int(text_tensor[0].ne(0).sum())  # 读取首条工单的真实 token 数量。
semantic_text = text_tensor[:1, :semantic_length]  # 构造不含任何右侧 PAD 的同语义 TextCNN 输入。
long_padded_text = F.pad(semantic_text, (0, 7), value=0)  # 为同一工单额外追加七个 PAD 位置。
with torch.no_grad():  # 关闭补齐不变性探针的梯度记录。
    short_text_logits, short_text_features, _ = textcnn(semantic_text, return_features=True)  # 计算无额外补齐时的 logits 与池化特征。
    long_text_logits, long_text_features, _ = textcnn(long_padded_text, return_features=True)  # 计算追加七个 PAD 后的同语义结果。
textcnn_padding_logit_error = float((short_text_logits - long_text_logits).abs().max())  # 量化 TextCNN 对补齐长度的 logits 偏差。
textcnn_padding_feature_error = float((short_text_features - long_text_features).abs().max())  # 量化 TextCNN 核心池化特征偏差。
saved_textcnn_pad = textcnn.embedding[0].detach().clone()  # 保存 TextCNN 当前随机 PAD 参数行。
with torch.no_grad():  # 关闭修改 PAD 哨兵参数时的梯度记录。
    reference_text_logits = textcnn(long_padded_text)  # 保存修改 PAD 参数前的同语义 logits。
    textcnn.embedding[0].fill_(9999.0)  # 把 PAD 参数改成巨大哨兵值以测试 forward 是否真正归零。
    mutated_text_logits = textcnn(long_padded_text)  # 在巨大 PAD 参数下重新执行 TextCNN forward。
    textcnn.embedding[0].copy_(saved_textcnn_pad)  # 恢复 PAD 参数避免影响后续实验状态。
textcnn_pad_parameter_error = float((reference_text_logits - mutated_text_logits).abs().max())  # 计算修改 PAD 参数造成的输出偏差。
textcnn.zero_grad()  # 清空训练阶段残留梯度以执行独立 PAD 梯度探针。
textcnn_probe_loss = F.cross_entropy(textcnn(long_padded_text), target_tensor[:1])  # 对含大量 PAD 的真实样本计算探针损失。
textcnn_probe_loss.backward()  # 把分类误差反传到词向量参数表。
textcnn_pad_gradient_norm = float(textcnn.embedding.grad[0].norm())  # 读取 TextCNN embedding 第零行的真实梯度范数。
long_padded_hierarchy = F.pad(hierarchy_tensor[:1], (0, 4, 0, 2), value=0)  # 为同一 HAN 文档追加四个词位和两个空句。
with torch.no_grad():  # 关闭 HAN 补齐不变性探针的梯度记录。
    base_han_logits, base_word_attention, base_sentence_attention = han(hierarchy_tensor[:1], return_attention=True)  # 计算原始层级输入的结果。
    padded_han_logits, padded_word_attention, padded_sentence_attention = han(long_padded_hierarchy, return_attention=True)  # 计算增加词句 PAD 后的结果。
    empty_document = torch.zeros((1, 2, 4), dtype=torch.long)  # 构造两句都完全由 PAD 组成的全空文档。
    empty_han_logits, empty_word_attention, empty_sentence_attention = han(empty_document, return_attention=True)  # 对全空文档执行真实 HAN 前向并收集两级权重。
han_padding_logit_error = float((base_han_logits - padded_han_logits).abs().max())  # 量化 HAN logits 对补齐长度的偏差。
han_word_attention_error = float((base_word_attention - padded_word_attention[:, :base_word_attention.shape[1], :base_word_attention.shape[2]]).abs().max())  # 比较全部有效词位置的注意力。
han_sentence_attention_error = float((base_sentence_attention - padded_sentence_attention[:, :base_sentence_attention.shape[1]]).abs().max())  # 比较全部有效句位置的注意力。
added_sentence_start = hierarchy_tensor.shape[1]  # 记录额外全 PAD 句在补齐文档中的起始位置。
padded_empty_word_attention_max = float(padded_word_attention[:, added_sentence_start:, :].abs().max())  # 检查正常文档新增空句内部的 PAD 词权重最大值。
empty_document_word_attention_max = float(empty_word_attention.abs().max())  # 检查全空文档的词级 PAD 权重最大值。
empty_document_sentence_attention_max = float(empty_sentence_attention.abs().max())  # 检查全空文档的句级 PAD 权重最大值。
empty_document_outputs_finite = bool(torch.isfinite(empty_han_logits).all() and torch.isfinite(empty_word_attention).all() and torch.isfinite(empty_sentence_attention).all())  # 验证全空输入不会产生 NaN 或无穷值。
saved_han_pad = han.embedding[0].detach().clone()  # 保存 HAN 当前随机 PAD 参数行。
with torch.no_grad():  # 关闭修改 HAN PAD 哨兵参数时的梯度记录。
    reference_han_logits = han(long_padded_hierarchy)  # 保存修改 PAD 参数前的层级 logits。
    han.embedding[0].fill_(-9999.0)  # 把 HAN PAD 参数改成巨大负哨兵值。
    mutated_han_logits = han(long_padded_hierarchy)  # 在巨大 PAD 参数下重新执行 HAN forward。
    han.embedding[0].copy_(saved_han_pad)  # 恢复 HAN PAD 参数避免改变模型制品。
han_pad_parameter_error = float((reference_han_logits - mutated_han_logits).abs().max())  # 计算修改 HAN PAD 参数造成的输出偏差。
han.zero_grad()  # 清空训练阶段残留梯度以执行独立 PAD 梯度探针。
han_probe_loss = F.cross_entropy(han(long_padded_hierarchy), target_tensor[:1])  # 对含额外空句的真实样本计算探针损失。
han_probe_loss.backward()  # 把分类误差反传到 HAN 词向量表。
han_pad_gradient_norm = float(han.embedding.grad[0].norm())  # 读取 HAN embedding 第零行的真实梯度范数。
print("位置       正文句1  正文句2  PAD空句")  # 输出注意力失败案例权重表头。
print("未mask权重", [round(float(value), 4) for value in naive_attention[0]])  # 展示 PAD 吞掉大部分概率质量。
print("已mask权重", [round(float(value), 4) for value in fixed_attention[0]])  # 展示 PAD 权重被严格归零。
print(f"聚合值：错误={float(naive_summary):.3f}，修复={float(fixed_summary):.3f}")  # 量化注意力 padding 污染前后的表示差异。
print("模型     补齐logit差  核心中间量差  改PAD参数差  PAD梯度范数")  # 输出真实模型 PAD 不变性结果表头。
print(f"TextCNN  {textcnn_padding_logit_error:.8f}    {textcnn_padding_feature_error:.8f}      {textcnn_pad_parameter_error:.8f}    {textcnn_pad_gradient_norm:.8f}")  # 展示卷积模型的四项数值证据。
han_attention_error = max(han_word_attention_error, han_sentence_attention_error)  # 汇总 HAN 有效词句注意力的最大偏差。
print(f"HAN      {han_padding_logit_error:.8f}    {han_attention_error:.8f}      {han_pad_parameter_error:.8f}    {han_pad_gradient_norm:.8f}")  # 展示层级模型的四项数值证据。
print(f"HAN新增全PAD句word权重最大值={padded_empty_word_attention_max:.8f}；全空文档word/sentence权重最大值={empty_document_word_attention_max:.8f}/{empty_document_sentence_attention_max:.8f}；输出有限={empty_document_outputs_finite}")  # 展示两种全空边界的零注意力与数值稳定性。

位置       正文句1  正文句2  PAD空句
未mask权重 [0.026, 0.0213, 0.9527]
已mask权重 [0.5498, 0.4502, 0.0]
聚合值：错误=94.344，修复=0.650
模型     补齐logit差  核心中间量差  改PAD参数差  PAD梯度范数
TextCNN  0.00000000    0.00000000      0.00000000    0.00000000
HAN      0.00000048    0.00000012      0.00000000    0.00000000
HAN新增全PAD句word权重最大值=0.00000000；全空文档word/sentence权重最大值=0.00000000/0.00000000；输出有限=True


## 生产差距与追问

真实系统还需要：按时间切分训练/验证/测试集；处理词表版本、OOV、超长工单和类别漂移；把 attention 当调试线索而非因果解释；报告 macro-F1、混淆矩阵、长度分桶、置信度校准与人工兜底率。TextCNN 适合低延迟短文本，HAN 适合保留段落层次的长工单；若使用预训练编码器，也仍要保留同样的数据、mask 与评估合同。

## 最小回归测试

In [7]:
assert text_tensor.shape[0] >= 5  # 保证教学案例数量满足可比较实验要求。
assert baseline_accuracy < textcnn_accuracy  # 保证手写 TextCNN 确实改善关键词规则反例。
assert textcnn_accuracy >= 0.9  # 保护 TextCNN 核心实现能拟合可解释小数据。
assert han_accuracy >= 0.9  # 保护 HAN 两级注意力实现能完成同一任务。
assert float(fixed_attention[0, 2]) == 0.0  # 保护句级 padding 不再获得注意力。
assert textcnn_padding_logit_error < 1e-7 and textcnn_padding_feature_error < 1e-7  # 保护 TextCNN 不利用补齐长度或部分 PAD 窗口。
assert han_padding_logit_error < 2e-6 and han_attention_error < 2e-6  # 在浮点舍入容差内保护 HAN 有效语义与注意力不随词句补齐长度改变。
assert textcnn_pad_parameter_error < 1e-7 and han_pad_parameter_error < 1e-7 and textcnn_pad_gradient_norm == 0.0 and han_pad_gradient_norm == 0.0 and padded_empty_word_attention_max == 0.0 and empty_document_word_attention_max == 0.0 and empty_document_sentence_attention_max == 0.0 and empty_document_outputs_finite  # 保护两套模型的 PAD 参数与梯度无法泄漏，并保证新增空句及全空文档的两级注意力全零且不产生 NaN。
print("最小回归测试通过：数据、训练、PAD查表、部分窗口、补齐不变性与零梯度均保持有效。")  # 输出集中测试结论。

最小回归测试通过：数据、训练、PAD查表、部分窗口、补齐不变性与零梯度均保持有效。
